In [1]:
import gc
gc.collect()

55

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import register_and_resample, sikit_to_just_data
from core.helper import load_NiFty_and_save_raw_data , copy_NiFty, patients_transform
from core.stat_calc import find_periods
from core.visualization_lib import show_transformation
from core.helper import transform_step_list_to_dictioanry

from core.transformers.nifti_to_raw_transformer import nifti_to_raw_transformer
from core.transformers.anatomy_fill_transformer import anatomy_fill_transformer
from core.transformers.crop_transformer import non_weighted_crop_transformer
from core.transformers.resample_transformer import resample_transformer


In [ ]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessing_steps_list = [
    ('start', ' nifty'),
    ('resampling','resampled')
    ('nifty_to_raw', 'raw'),
    ('filling_anatomy_gaps', 'anatomy_gap_filled'),
    ('cropping', 'cropped'),

]

preprocessed_steps = transform_step_list_to_dictioanry(preprocessing_steps_list)

crop_size = (160,160,24)
target_spacing=(0.8, 0.8, 3.5)

In [4]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

copy data to preprocess folder

In [5]:
copy_NiFty(orginal_data_folder, file_manager,channels, filter=['3322'], step=preprocessed_steps['nifi_to_raw']['start'])
patients = file_manager.get_file_names()

<h2>resampling</h2>


In [ ]:
resample_transformer = resample_transformer(target_spacing=target_spacing, is_label=False)


start_step = preprocessed_steps['resampling']['start']
end_step = preprocessed_steps['resampling']['end']
patients_transform(file_manager,)


NameError: name 'resample_transformer' is not defined

transform to raw

In [6]:
start_step = preprocessed_steps['nifi_to_raw']['start']
end_step = preprocessed_steps['nifi_to_raw']['end']

to_raw_transform = nifti_to_raw_transformer()

patients_transform(file_manager, start_step, end_step, to_raw_transform)

dispaly

In [7]:
folder_shower(file_manager,'1_raw',normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…

remember about allingning

<h2>Working with holes in prostate layer</h2>

In [8]:
from core.anatomy_gap_fixer import find_gaps_in_anatomy
from core.anatomy_gap_fixer import fix_patient_anatomy

start_step = preprocessed_steps['filling_anatomy_gaps']['start']
end_step = preprocessed_steps['filling_anatomy_gaps']['end']

checking out outliers

In [9]:
# outliers, _ = find_gaps_in_anatomy(patients,file_manager,step = start_step)
# for outlier in outliers:
#     print(outlier)
#     prostate = file_manager.load_file(start_step, outlier)['anatomy'] 
#     valid_layers = np.any(prostate == 1, axis=(0, 1))
#     true_indices = np.where(valid_layers)[0]
#     periods = find_periods(true_indices)
#     print(len(periods))
#     print(periods)

FIX outliers

In [10]:
gap_fill_transformer = anatomy_fill_transformer()
patients_transform(file_manager, start_step, end_step,gap_fill_transformer)

20  22


<h2>CROPPING!!!</h2>

Cutting pictures into correct sizes

two ways of centering

finding maximum prostate dimentions

In [11]:
from core.stat_calc import find_patients_max_prostate_sizes

start_step = preprocessed_steps['cropping']['start']
end_step = preprocessed_steps['cropping']['end']

maximum_prostate_size = find_patients_max_prostate_sizes(patients,file_manager, step = start_step)
print(maximum_prostate_size)

[67, 63, 16]


center cropping

THERE IS NO PADDING!!!!

In [12]:
nw_crop_transformer = non_weighted_crop_transformer(crop_size)
patients_transform(file_manager, start_step, end_step, nw_crop_transformer)

In [13]:
folder_shower(file_manager,normalizer=normalize_volume,step = start_step)
folder_shower(file_manager,step = end_step,normalizer=normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…